# Q-Mini Add/Sub 2M — Complete Model Project First (No Dataset, No Training)

Run the single code cell. It writes a complete model project bundle to Google Drive: all Add/Sub Python/native source files and training/inference entry points, the exact model config, deterministic UTF-8 byte tokenizer, the complete initialized step-zero tensor state, a per-parameter shape/offset layout, file hashes, and a model card.

It uses CPU only. It does not download datasets, ask for license approval, initialize an optimizer, or train. It preserves the existing Hugging Face cache, approval records, and old model checkpoints.

The dataset-independent tokenizer has 256 byte symbols plus 8 control tokens. With the 2M architecture profile, the model has **1,887,644 actual parameters** under the 2,000,000 limit. The saved source package is then imported in a separate Python process from the Drive copy itself and the model is forward-tested. The source, tokenizer, config and parameter IDs must remain fixed once training begins.

In [ ]:
# Q-MINI ADD/SUB 2M — MODEL-FIRST INITIALIZATION ONLY
# No dataset download. No license prompt. No training. CPU only.

from google.colab import drive
drive.mount("/content/drive")

import subprocess
import sys
from pathlib import Path

import torch

REPO_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Q-mini-light-0.7M")
CPU_THREADS = 2
DEVICE = "cpu"

def run(command, *, cwd=None):
    print("\n>>>", " ".join(str(part) for part in command), flush=True)
    subprocess.run([str(part) for part in command], cwd=str(cwd) if cwd else None, check=True)

print("[1/4] Getting latest merged Q-Mini source...")
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository. Inspect it; this cell will not delete it.")
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], cwd=REPO_DIR)

REPO_DIR = REPO_DIR.resolve()
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository commit:", REPO_COMMIT)

print("[2/4] Installing only the model/tokenizer dependencies (preserving Colab PyTorch)...")
run([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24", "tokenizers>=0.15"])
run([sys.executable, "-m", "pip", "install", "--no-deps", "-q", "-e", str(REPO_DIR)])

if str(REPO_DIR) in sys.path:
    sys.path.remove(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
import importlib
importlib.invalidate_caches()
import qmini_addsub
from qmini_addsub.initialize import initialize_model_artifacts, verify_model_bundle

package_path = Path(qmini_addsub.__file__).resolve()
if REPO_DIR not in package_path.parents:
    raise RuntimeError(f"Imported Q-Mini from the wrong checkout: {package_path}")

torch.set_num_threads(CPU_THREADS)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE, "(GPU is not used)")
print("CPU threads:", torch.get_num_threads())
print("Package:", package_path)

print("[3/4] Creating/verifying the complete model in Google Drive...")
WORKSPACE_ROOT = Path("/content/drive/MyDrive/Q mini 2 light/trainability_addsub_2m_v1")
PROFILE_PATH = REPO_DIR / "configs" / "qmini_addsub_2m.json"
manifest = initialize_model_artifacts(
    WORKSPACE_ROOT,
    PROFILE_PATH,
    threads=CPU_THREADS,
    source_root=REPO_DIR,
    source_commit=REPO_COMMIT,
)
verified = verify_model_bundle(WORKSPACE_ROOT)

ARTIFACT_DIR = WORKSPACE_ROOT / "artifacts"
print("[4/4] MODEL INITIALIZATION VERIFIED")
print("Model stage:", verified["stage"])
print("Parameters:", f"{verified['parameter_count']:,}", "/ 2,000,000")
print("Vocabulary:", verified["tokenizer"]["vocabulary_size"])
print("Context:", verified["architecture"]["context_length"])
print("Tokenizer:", ARTIFACT_DIR / "tokenizer.json")
print("Exact config:", ARTIFACT_DIR / "model_config.json")
print("Initialized weights:", ARTIFACT_DIR / "initial_model.pt")
print("Parameter layout:", ARTIFACT_DIR / "parameter_layout.json")
print("Self-contained Python source:", ARTIFACT_DIR / "source_code")
print("Source files copied:", verified["source_snapshot"]["file_count"])
print("Python source files:", verified["source_snapshot"]["python_file_count"])
print("Model manifest:", ARTIFACT_DIR / "model_manifest.json")
print("Model card:", ARTIFACT_DIR / "MODEL_CARD.md")
print("Independent source import:", manifest["source_verification"]["isolated_source_import"])
print("Drive workspace:", WORKSPACE_ROOT)
print("Dataset downloaded by this cell: NO")
print("Training started by this cell: NO")
print("Next step: use the separate QMini_AddSub_2M_One_Click.ipynb only when you are ready to prepare data and train this exact initialized model.")